In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ar6_ch6_rcmipfigs.constants import INPUT_DATA_DIR_BADC, INPUT_DATA_DIR, \
    OUTPUT_DATA_DIR, RESULTS_DIR
from ar6_ch6_rcmipfigs.utils.badc_csv import read_csv_badc

In [ ]:
fn_concentrations = INPUT_DATA_DIR / 'historical_delta_GSAT/LLGHG_history_AR6_v9_updated.xlsx'
path_emissions = INPUT_DATA_DIR / 'historical_delta_GSAT/CEDS_v2021-02-05_emissions/'

# file path table of ERF 2019-1750
fp_collins = OUTPUT_DATA_DIR / 'fig6_12_ts15_historic_delta_GSAT/table_mean_thornhill_collins_orignames.csv'

In [ ]:
fl_CEDS = list(path_emissions.glob('*global_CEDS_emissions_by_sector_2021_02_05.csv'))

In [ ]:
fn_output_ERF = OUTPUT_DATA_DIR / 'fig6_12_ts15_historic_delta_GSAT/hist_ERF_est.csv'
fn_output_ERF_2019 = OUTPUT_DATA_DIR / 'fig6_12_ts15_historic_delta_GSAT/2019_ERF_est.csv'
fn_output_decomposition = OUTPUT_DATA_DIR / 'fig6_12_ts15_historic_delta_GSAT/hist_ERF_est_decomp.csv'

In [ ]:

df_conc = pd.read_excel(fn_concentrations, header=22, index_col=0, engine='openpyxl')
# adds unnecessary row with nans and unnamed columns
df_conc = df_conc.loc[1750:2019]
unnamed = [c for c in df_conc.columns if 'Unnamed:' in c]
df_conc = df_conc.drop(unnamed, axis=1)

# For C8F18 there appears to be an error in the spreadsheet where 2015 is entered as zero, presumably 0.09 but treat as missing
df_conc.loc[2015, 'C8F18'] = np.nan

# datetime index
df_conc.index = pd.to_datetime(df_conc.index.astype(int), format='%Y')

# resample to yearly, i.e. NaNs will be filled between 1750 and 1850:
df_conc = df_conc.resample('Y').first()  # .interpolate()
# Interpolate:
df_conc = df_conc.interpolate(method='linear',
                              limit_area='inside')  # 'inside' only fill values with valid on both sides.
# reset index to year
df_conc.index = df_conc.index.year
df_conc

In [ ]:
[c for c in df_conc.columns if 'CFC' in c]

In [ ]:
list_df_em = []
units_dic = {}
for fn in fl_CEDS:
    _df = pd.read_csv(fn)
    u_em = _df['em'].unique()
    if len(u_em) > 1:
        print('double check, emission labels :')
        print(u_em)
    _em = u_em[0]
    u_units = _df['units'].unique()
    if len(u_units) > 1:
        print('double check, units labels :')
        print(u_units)
    _un = u_units[0]
    _df_s = _df.drop(['em', 'sector', 'units'], axis=1).sum()
    _df_s.index = pd.to_datetime(_df_s.index, format='X%Y').year
    _df = pd.DataFrame(_df_s, columns=[_em])
    units_dic[_em] = _un
    list_df_em.append(_df)

In [ ]:
df_emissions = pd.concat(list_df_em, axis=1)

In [ ]:
df_emissions

In [ ]:
units_dic

In [ ]:
df_collins = pd.read_csv(fp_collins, index_col=0)
df_collins.index = df_collins.index.rename('emission_experiment')

In [ ]:
df_collins  # .columns

In [ ]:
df_collins.sum()  # .columns

In [ ]:
forcing_total_collins = df_collins.sum(axis=1)  # ['Total']
forcing_total_collins

In [ ]:
def scale_ERF(forcing_tot, df_agent, spec_lab, spec_cmip, end_year=2019, base_year=1750):
    """
    Scale ERF forcing in the end year (2019) by the concentrations/emissions each year
    divided by the concentration/emission in the end year (relative to the base year).
    :param forcing_tot:
    :param df_agent:
    :param spec_lab:
    :param spec_cmip:
    :param end_year:
    :param base_year:
    :return:
    """
    delta_spec_end_year = df_agent[spec_lab].loc[end_year] - df_agent[spec_lab].loc[base_year]  # 2019
    delta_spec = df_agent[spec_lab] - df_agent[spec_lab].loc[base_year]  # 1750-2019
    aerchemmip_endyear_forcing = forcing_tot[spec_cmip]  # from Bill collins
    erf_spec = aerchemmip_endyear_forcing * delta_spec / delta_spec_end_year  # scale by concentrations
    return erf_spec

In [ ]:
fig, ax = plt.subplots(figsize=[10, 5])

ERFs = {}
for spec in ['CO2', 'N2O', 'CH4']:
    forcing_spec = scale_ERF(forcing_total_collins, df_conc, spec, spec)  # # scale by concentrations

    print(spec)
    forcing_spec.plot(label=spec)
    ERFs[spec] = forcing_spec

spec = 'NOx'
for spec in ['NOx', 'SO2', 'BC', 'OC', 'NH3']:
    forcing_spec = scale_ERF(forcing_total_collins, df_emissions, spec, spec)  # scale by emissions

    ERFs[spec] = forcing_spec

    forcing_spec.plot(label=spec)

# VOC: scale with CO emissions because these are mostly the same
spec = 'CO'

forcing_spec = scale_ERF(forcing_total_collins, df_emissions, spec, 'VOC')  # scale by concentrations

ERFs['VOC'] = forcing_spec

forcing_spec.plot(label=spec)

plt.ylabel('W m$^{-2}$')

plt.legend(loc='upper left')

In [ ]:
fp_hodnebrog = INPUT_DATA_DIR_BADC / 'hodnebrog_tab3.csv'
#fp_hodnebrog = INPUT_DATA_DIR / 'hodnebrog_tab3.csv'

In [ ]:
df_hodnebrog = read_csv_badc(fp_hodnebrog, index_col=[0, 1], header=[0, 1])
df_HFC = df_hodnebrog.loc[('Hydrofluorocarbons',)]
df_HFC

In [ ]:
RE_df = df_HFC['RE (Wm-2ppb-1)'].transpose()
# RE_df = RE_df.reset_index().rename({'level_1':'Species'},axis=1).set_index('Species').drop('level_0', axis=1)
RE_df  # .transpose().loc['This work']*

In [ ]:
df_conc[RE_df.columns] - df_conc[RE_df.columns].loc[1750]

In [ ]:
ERF_HFCs = (df_conc[RE_df.columns] - df_conc[RE_df.columns].loc[1750]) * RE_df.loc['This work'] * 1e-3  # ppt to ppb
ERF_HFCs['HFCs'] = ERF_HFCs.sum(axis=1)
ERF_HFCs

In [ ]:
for c in ERF_HFCs.columns[:-1]:
    ERF_HFCs[c].plot(label=c)
ERF_HFCs.sum(axis=1).plot(label='HFCs', color='k', linewidth=3)
plt.legend()
plt.ylabel('W/m2')

In [ ]:
# We use CFC-12 emissions for HC
spec = 'CFC-12'
forcing_HC = scale_ERF(forcing_total_collins, df_conc, spec, 'HC')  # scale by concentrations

ERFs['HC'] = forcing_HC + ERF_HFCs['HFCs']

ERF_HFCs['HFCs']

In [ ]:
df_ERF = pd.concat(ERFs, axis=1)  # ['CO2'].loc[1752]
df_ERF.columns

In [ ]:
forcing_HC[2019]

In [ ]:
df_collins['HFCs'] = 0
df_collins.loc['HC', 'HFCs'] = ERF_HFCs['HFCs'][2019]
df_collins

In [ ]:
df_collins.sum(axis=0)

In [ ]:
df_ERF

In [ ]:
df_coll_t = df_collins.transpose()
if 'Total' in df_coll_t.index:
    df_coll_t = df_coll_t.drop('Total')
# scale by total:
scale = df_coll_t.sum()
# normalized ERF: 
df_col_normalized = df_coll_t / scale
#
df_col_normalized.transpose().plot.barh(stacked=True)
plt.legend(bbox_to_anchor=(1, 1))

In [ ]:
fn_output_ERF

In [ ]:
fn_output_ERF.parent.mkdir(parents=True, exist_ok=True)
df_ERF.to_csv(fn_output_ERF)
df_col_normalized.to_csv(fn_output_decomposition)
df_collins.to_csv(fn_output_ERF_2019)

In [ ]:
df_ERF